# Generador de dataset v2 (Colab A100): frase chilena → frase neutralizada

Versión pensada para una **A100 en Colab** con un modelo de ~30B servido con **vLLM**. Cambios respecto a `models/mistral7B`:

1. **Etapa A: ejemplos reales.** Cada definición del diccionario trae un ejemplo escrito por un chileno. El modelo solo lo limpia (quita aclaraciones entre paréntesis) y lo neutraliza. La frase chilena es auténtica.
2. **Etapa B: frases sintéticas por definición** (no por término), con ejemplos completos en el prompt y temperatura más baja.
3. **Etapa C: juez.** El modelo evalúa cada par (¿usa la expresión con el sentido correcto?, ¿la neutral dice lo mismo?, ¿es gramatical?, ¿quedan chilenismos?). Solo pasan los pares aprobados en todo.
4. Filtro léxico corregido (ignora palabras como *ser*/*estar*/*andar* y usa prefijos más largos), que queda como columna informativa.

Cada etapa guarda su CSV y se salta si ya existe, así que se puede cortar y retomar. Las salidas quedan en `data/entrenamiento/v2/<modelo>/`.

**Para comparar modelos:** pon `LIMITE = 40`, corre todo con un modelo, cambia `MODEL_ID`, reinicia el entorno y vuelve a correr. La última celda arma una planilla ciega para evaluar a mano.

In [ ]:
# [COLAB] Instalación (tarda unos minutos). Si Colab pide reiniciar el entorno, reinícialo y sigue desde la celda siguiente.
%pip install -q vllm pandas

In [ ]:
# [COLAB] Verificación de la GPU
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory / 1e9:.0f} GB")

In [ ]:
# Configuración
MODEL_ID = "Qwen/Qwen3-32B-AWQ"   # ver la tabla de modelos en la nota final
MAX_MODEL_LEN = 4096
GPU_MEM = 0.90

MAX_DEFS = 3          # definiciones por expresión (las de más votos) para la etapa B
K_SINT = 3            # frases sintéticas por definición
T_GEN = 0.6           # temperatura de generación (etapas A y B)
T_JUEZ = 0.0          # el juez responde de forma determinista
SEED = 42
LIMITE = None         # p. ej. 40 expresiones para comparar modelos; None = todas

REPO_URL = "https://github.com/matiaraos/NLP-ChiNeutro.git"   # si es privado: https://<token>@github.com/...
GUARDAR_EN_DRIVE = False   # True copia las salidas a Google Drive al final

In [ ]:
# Rutas: busca el repo hacia arriba; en Colab, si no está, lo clona
import os, re, subprocess
from pathlib import Path

ARCHIVO = "scraping/data/rag_documentos.jsonl"


def encontrar_raiz():
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / ARCHIVO).is_file():
            return carpeta
    destino = Path("/content/NLP-ChiNeutro")
    if not (destino / ARCHIVO).is_file():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destino)], check=True)
    return destino


RAIZ = encontrar_raiz()
SLUG = re.sub(r"[^A-Za-z0-9._-]+", "_", MODEL_ID.split("/")[-1])
SUFIJO = f"_lim{LIMITE}" if LIMITE else ""
DIR_SALIDA = RAIZ / "data/entrenamiento/v2" / f"{SLUG}{SUFIJO}"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)
RUTA_A = DIR_SALIDA / "etapa_a_ejemplos_reales.csv"
RUTA_B = DIR_SALIDA / "etapa_b_sinteticos.csv"
RUTA_C = DIR_SALIDA / "etapa_c_juez.csv"
RUTA_FINAL = DIR_SALIDA / "dataset_final.csv"
print("Raíz:", RAIZ, "\nSalidas:", DIR_SALIDA)

In [ ]:
# Lectura del jsonl
import json
import random
import pandas as pd

with open(RAIZ / ARCHIVO, encoding="utf-8") as f:
    docs = [json.loads(l) for l in f if l.strip()]


def votos(x):
    try:
        return int(x.get("votos") or 0)
    except (TypeError, ValueError):
        return 0


for d in docs:
    d["definiciones"] = sorted([x for x in d.get("definiciones", []) if (x.get("definicion") or "").strip()],
                               key=votos, reverse=True)
docs = [d for d in docs if d["definiciones"]]
if LIMITE:
    docs = random.Random(SEED).sample(docs, min(LIMITE, len(docs)))   # misma muestra para todos los modelos
DOC = {d["clave"]: d for d in docs}
print(f"Expresiones: {len(docs)} | definiciones: {sum(len(d['definiciones']) for d in docs)}")

In [ ]:
# Utilidades: normalización, filtro léxico (corregido) y parser de JSON
import unicodedata

PALABRAS_VACIAS = {"ser", "estar", "andar", "ando", "anda", "estoy", "esta", "tener", "tengo", "hacer",
                   "dar", "echar", "el", "la", "los", "las", "un", "una", "de", "del", "al", "a", "en",
                   "que", "se", "te", "me", "le", "lo", "y", "o", "pa", "con", "por", "para", "como"}


def norm(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z0-9ñ\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def formas_de(doc):
    return {norm(f) for f in [doc["clave"], doc["termino"], *doc.get("variantes", [])] if norm(f)}


def contiene_expresion(frase, doc, flexible=True):
    """La frase contiene la expresión o una variante. En modo flexible acepta conjugaciones:
    cada palabra de contenido debe compartir prefijo (4 a 6 letras según su largo) con alguna palabra de la frase."""
    f = norm(frase)
    if any(f" {forma} " in f" {f} " for forma in formas_de(doc)):
        return True
    if not flexible:
        return False
    palabras = f.split()
    for forma in formas_de(doc):
        toks = [t for t in forma.split() if t not in PALABRAS_VACIAS and len(t) > 2]
        if toks and all(any(w.startswith(t[:max(4, min(len(t) - 2, 6))]) for w in palabras) for t in toks):
            return True
    return False


def limpiar_salida(texto):
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.S).strip()


def extraer_objetos(texto):
    texto = limpiar_salida(texto)
    ini, fin = texto.find("["), texto.rfind("]")
    if ini != -1 and fin > ini:
        try:
            data = json.loads(texto[ini:fin + 1])
            if isinstance(data, list):
                objs = [x for x in data if isinstance(x, dict)]
                if objs:
                    return objs
        except json.JSONDecodeError:
            pass
    objs = []
    for m in re.finditer(r"\{[^{}]*\}", texto):
        try:
            o = json.loads(m.group(0))
            if isinstance(o, dict):
                objs.append(o)
        except json.JSONDecodeError:
            continue
    return objs


def limpio(s):
    return re.sub(r"\s+", " ", str(s or "")).strip()

In [ ]:
# Prompts
EJEMPLOS_GUIA = """Ejemplos de pares bien hechos (con otras expresiones; no los copies):
- Expresión "al tiro" (enseguida): {"chilena": "Espérame un poco, voy al tiro.", "neutral": "Espérame un poco, voy enseguida."}
- Expresión "estar pato" (no tener dinero): {"chilena": "Este finde no salgo, cabros, estoy pato.", "neutral": "Este fin de semana no salgo, chicos, no tengo dinero."}
- Expresión "pega" (trabajo): {"chilena": "Mi pololo me fue a buscar a la pega, ¿cachai?", "neutral": "Mi novio me fue a buscar al trabajo, ¿entiendes?"}
Fíjate en que la versión neutral reemplaza TODOS los chilenismos (cabros, pololo, cachai), no solo la expresión pedida, y conserva persona, tiempo verbal, información y tono."""

PROMPT_A = """Eres un lingüista experto en el español de Chile y en español neutro.

Esta frase la escribió un chileno en un diccionario colaborativo como ejemplo de la expresión "{termino}", con este sentido: {definicion}

FRASE ORIGINAL: {ejemplo}

{guia}

Tarea:
1. "chilena": deja la frase lista como ejemplo de habla chilena real. Quita aclaraciones entre paréntesis dirigidas al lector (por ejemplo "carrete (fiesta)" queda "carrete") y corrige solo errores de tipeo evidentes. Conserva la expresión, el resto del vocabulario chileno y el registro tal como están. Si es muy larga, quédate con la parte que usa la expresión (máximo 40 palabras).
2. "neutral": reescribe la frase chilena en español neutro y estándar, comprensible para cualquier hispanohablante: reemplaza la expresión y todos los demás chilenismos, conjugaciones chilenas (estay, cachai, erí) y muletillas (po, oe, weón) por equivalentes neutros. Conserva el significado, la información, la persona gramatical, el tiempo verbal y el tono; si la frase es grosera, usa un equivalente informal o grosero de uso general en vez de suavizarla.

Si la frase no usa realmente la expresión con ese sentido o es incomprensible, responde exactamente {{"descartar": true}}.
Responde SOLO con un objeto JSON: {{"chilena": "...", "neutral": "..."}}"""

PROMPT_B = """Eres un lingüista experto en el español de Chile y en español neutro.

EXPRESIÓN: {termino}
{variantes}SENTIDO: {definicion}
EJEMPLO REAL DE USO: {ejemplo}

{guia}

Tarea: escribe {k} pares de frases nuevas que usen la expresión con ESE sentido.
- "chilena": una frase natural, como la diría un chileno en una conversación, que use la expresión (puedes conjugarla o ajustar género y número). Debe sonar a habla real, sin explicar la expresión dentro de la frase. No copies el ejemplo.
- "neutral": la misma frase en español neutro y estándar, sin la expresión ni otros chilenismos, con el mismo significado, información, persona, tiempo verbal y tono.
Varía la situación, la persona gramatical y el largo (entre 6 y 25 palabras). No uses comillas dobles dentro de las frases.
Responde SOLO con un arreglo JSON: [{{"chilena": "...", "neutral": "..."}}, ...]"""

PROMPT_JUEZ = """Evalúa un par de entrenamiento para un sistema que traduce español de Chile a español neutro. Sé estricto.

EXPRESIÓN: {termino}
SENTIDO ESPERADO: {definicion}
FRASE CHILENA: {chilena}
FRASE NEUTRAL: {neutral}

Responde SOLO con un objeto JSON con estas claves:
- "usa_expresion": true si la frase chilena contiene la expresión (o una forma conjugada o variante de ella).
- "sentido_correcto": true si la usa con el sentido esperado (o uno muy cercano) y de una forma que un chileno diría de verdad.
- "chilena_gramatical": true si la frase chilena es coherente y está bien construida para el habla informal chilena.
- "neutral_equivalente": true si la frase neutral dice lo mismo: misma información, persona, tiempo verbal e intención, sin agregar ni quitar nada importante.
- "neutral_sin_chilenismos": true si la frase neutral no tiene chilenismos ni modismos locales y la entendería cualquier hispanohablante.
- "neutral_gramatical": true si la frase neutral es gramatical y natural.
- "comentario": una frase breve explicando el principal problema, o "" si no hay."""

CRITERIOS = ["usa_expresion", "sentido_correcto", "chilena_gramatical",
             "neutral_equivalente", "neutral_sin_chilenismos", "neutral_gramatical"]


def variantes_txt(doc):
    otras = [v for v in doc.get("variantes", []) if v != doc["termino"]]
    return f"VARIANTES: {', '.join(otras)}\n" if otras else ""


def corto(s, n):
    s = limpio(s)
    return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + "…"

In [ ]:
# [GPU] Carga del modelo con vLLM
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM, seed=SEED)

In [ ]:
# [GPU] Llamada al modelo en lote (vLLM agenda todo el lote solo)
def chatear(prompts, temperatura, max_tokens):
    mensajes = [[{"role": "user", "content": p}] for p in prompts]
    params = SamplingParams(temperature=temperatura, top_p=0.9 if temperatura > 0 else 1.0,
                            max_tokens=max_tokens, seed=SEED)
    extra = {"chat_template_kwargs": {"enable_thinking": False}} if "qwen3" in MODEL_ID.lower() else {}
    salidas = llm.chat(mensajes, params, use_tqdm=True, **extra)
    return [limpiar_salida(o.outputs[0].text) for o in salidas]

In [ ]:
# Etapa A: limpiar y neutralizar los ejemplos reales del diccionario
def etapa_a():
    if RUTA_A.exists():
        print("Etapa A ya existe, se salta:", RUTA_A)
        return pd.read_csv(RUTA_A)
    tareas = [(d, x) for d in docs for x in d["definiciones"] if limpio(x.get("ejemplo"))]
    prompts = [PROMPT_A.format(termino=d["termino"], definicion=corto(x["definicion"], 300),
                               ejemplo=corto(x["ejemplo"], 400), guia=EJEMPLOS_GUIA) for d, x in tareas]
    salidas = chatear(prompts, T_GEN, 300)
    filas, descartadas = [], 0
    for (d, x), texto in zip(tareas, salidas):
        objs = extraer_objetos(texto)
        o = objs[0] if objs else {}
        ch, ne = limpio(o.get("chilena")), limpio(o.get("neutral"))
        if o.get("descartar") or not ch or not ne or norm(ch) == norm(ne):
            descartadas += 1
            continue
        filas.append({"clave": d["clave"], "termino": d["termino"], "def_id": x.get("id"),
                      "origen": "ejemplo_real", "frase_chilena": ch, "frase_neutralizada": ne})
    df = pd.DataFrame(filas)
    df.to_csv(RUTA_A, index=False)
    print(f"Etapa A: {len(df)} pares | descartados o mal formados: {descartadas}")
    return df


df_a = etapa_a()

In [ ]:
# Etapa B: frases sintéticas por definición
def etapa_b():
    if RUTA_B.exists():
        print("Etapa B ya existe, se salta:", RUTA_B)
        return pd.read_csv(RUTA_B)
    tareas = [(d, x) for d in docs for x in d["definiciones"][:MAX_DEFS]]
    prompts = [PROMPT_B.format(termino=d["termino"], variantes=variantes_txt(d),
                               definicion=corto(x["definicion"], 300), ejemplo=corto(x.get("ejemplo"), 300),
                               guia=EJEMPLOS_GUIA, k=K_SINT) for d, x in tareas]
    salidas = chatear(prompts, T_GEN, 90 * K_SINT + 100)
    filas = []
    for (d, x), texto in zip(tareas, salidas):
        ejemplo = norm(x.get("ejemplo") or "")
        for o in extraer_objetos(texto)[:K_SINT]:
            ch, ne = limpio(o.get("chilena")), limpio(o.get("neutral"))
            if not ch or not ne or norm(ch) == norm(ne):
                continue
            if not (3 <= len(ch.split()) <= 45 and 3 <= len(ne.split()) <= 45):
                continue
            if len(ejemplo.split()) >= 4 and (norm(ch) in ejemplo or ejemplo in norm(ch)):
                continue   # copia del ejemplo real
            filas.append({"clave": d["clave"], "termino": d["termino"], "def_id": x.get("id"),
                          "origen": "sintetico", "frase_chilena": ch, "frase_neutralizada": ne})
    df = pd.DataFrame(filas)
    df.to_csv(RUTA_B, index=False)
    print(f"Etapa B: {len(df)} pares de {len(tareas)} definiciones (máx. {len(tareas) * K_SINT})")
    return df


df_b = etapa_b()

In [ ]:
# Etapa C: juez + filtro léxico + deduplicación
DEF_TXT = {x.get("id"): x["definicion"] for d in docs for x in d["definiciones"]}


def etapa_c(df):
    if RUTA_C.exists():
        print("Etapa C ya existe, se salta:", RUTA_C)
        return pd.read_csv(RUTA_C)
    df = df.copy()
    df["_n"] = df["frase_chilena"].map(norm)
    df = df.drop_duplicates("_n").drop(columns="_n").reset_index(drop=True)   # prioriza ejemplos reales (van primero)
    prompts = [PROMPT_JUEZ.format(termino=r.termino, definicion=corto(DEF_TXT.get(r.def_id, ""), 300),
                                  chilena=r.frase_chilena, neutral=r.frase_neutralizada) for r in df.itertuples()]
    salidas = chatear(prompts, T_JUEZ, 200)
    veredictos = []
    for texto in salidas:
        objs = extraer_objetos(texto)
        o = objs[0] if objs else {}
        veredictos.append({c: o.get(c) is True for c in CRITERIOS} | {"comentario_juez": limpio(o.get("comentario"))})
    df = pd.concat([df, pd.DataFrame(veredictos)], axis=1)
    df["aprobado_juez"] = df[CRITERIOS].all(axis=1)
    df["contiene_clave"] = [contiene_expresion(r.frase_chilena, DOC[r.clave]) for r in df.itertuples()]
    df["neutral_sin_clave"] = [not contiene_expresion(r.frase_neutralizada, DOC[r.clave], flexible=False)
                               for r in df.itertuples()]
    df["modelo"] = MODEL_ID
    df.to_csv(RUTA_C, index=False)
    return df


df_c = etapa_c(pd.concat([df_a, df_b], ignore_index=True))
final = df_c[df_c["aprobado_juez"] & df_c["neutral_sin_clave"]]
final.to_csv(RUTA_FINAL, index=False, encoding="utf-8-sig")

print(f"Pares evaluados: {len(df_c)} | aprobados: {len(final)} ({len(final) / len(df_c):.1%}) -> {RUTA_FINAL}")
print(f"Expresiones con al menos un par: {final['clave'].nunique()} de {len(docs)}")
print("\nTasa de aprobación por criterio y origen:")
print(df_c.groupby("origen")[CRITERIOS + ["aprobado_juez", "contiene_clave"]].mean().T.round(2))

In [ ]:
# Revisión: muestras aprobadas y rechazadas
pd.set_option("display.max_colwidth", None)
cols = ["origen", "termino", "frase_chilena", "frase_neutralizada"]
print("APROBADOS"); display(final.sample(min(10, len(final)), random_state=0)[cols])
rech = df_c[~df_c["aprobado_juez"]]
print("RECHAZADOS"); display(rech.sample(min(10, len(rech)), random_state=0)[cols + ["comentario_juez"]])

In [ ]:
# Comparación de modelos: planilla ciega con las corridas que existan en data/entrenamiento/v2/
# Usa las salidas del juez (antes de filtrar) para evaluar la generación, no al juez.
N_POR_MODELO = 40
corridas = sorted((RAIZ / "data/entrenamiento/v2").glob("*/etapa_c_juez.csv"))
print("Corridas encontradas:", [c.parent.name for c in corridas])
if len(corridas) >= 2:
    partes = []
    for c in corridas:
        x = pd.read_csv(c)
        x["corrida"] = c.parent.name
        partes.append(x.sample(min(N_POR_MODELO, len(x)), random_state=SEED))
    ciega = pd.concat(partes).sample(frac=1, random_state=SEED).reset_index(drop=True)
    ciega["id"] = range(1, len(ciega) + 1)
    for c in ["nota_uso_expresion", "nota_neutral", "nota_gramatica", "observaciones"]:
        ciega[c] = ""
    dir_eval = RAIZ / "data/entrenamiento/v2/evaluacion"
    dir_eval.mkdir(exist_ok=True)
    ciega[["id", "termino", "frase_chilena", "frase_neutralizada", "nota_uso_expresion", "nota_neutral",
           "nota_gramatica", "observaciones"]].to_csv(dir_eval / "planilla_ciega.csv", index=False, encoding="utf-8-sig")
    ciega[["id", "corrida", "origen", "aprobado_juez"]].to_csv(dir_eval / "clave_planilla.csv", index=False)
    print("Planilla:", dir_eval / "planilla_ciega.csv", "| clave (no abrir antes de evaluar):", dir_eval / "clave_planilla.csv")

In [ ]:
# [COLAB] Copiar salidas a Google Drive (opcional)
if GUARDAR_EN_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    destino = Path("/content/drive/MyDrive/NLP-ChiNeutro/v2")
    shutil.copytree(RAIZ / "data/entrenamiento/v2", destino, dirs_exist_ok=True)
    print("Copiado a", destino)

## Notas

**Modelos sugeridos** (revisen en Hugging Face si hay una versión más nueva de la misma familia):

| GPU | Modelo | Comentario |
|---|---|---|
| A100 40 GB | `Qwen/Qwen3-32B-AWQ` | Opción por defecto. ~19 GB de pesos, deja espacio para el caché. Se usa sin modo de razonamiento. |
| A100 40 GB | `Qwen/Qwen2.5-32B-Instruct-AWQ` | Alternativa sin modo de razonamiento. |
| A100 40 GB | Gemma 3 27B instruct (versión AWQ/GPTQ en 4 bits) | Muy bueno en multilingüe; requiere aceptar la licencia en HF. |
| A100 80 GB | `Qwen/Qwen2.5-72B-Instruct-AWQ` o Llama 3.3 70B en 4 bits | El mayor salto de calidad que cabe en una GPU. |

- **El juez es el mismo modelo que genera**, así que tiende a aprobar sus propios errores. Por eso conviene revisar a mano la planilla ciega. Si sobra GPU, se puede evaluar con un modelo distinto: reinicia el entorno, pon otro `MODEL_ID`, copia el `etapa_c_juez.csv` y vuelve a correr solo la etapa C sin el archivo previo.
- **Columnas de `dataset_final.csv`:** `clave`, `termino`, `def_id`, `origen` (`ejemplo_real` o `sintetico`), `frase_chilena`, `frase_neutralizada`, los seis criterios del juez, `comentario_juez`, `aprobado_juez`, `contiene_clave`, `neutral_sin_clave` y `modelo`.
- Para **rehacer una etapa**, borra su CSV (y los de las etapas siguientes).